In [0]:
%run ../0-common/env-config

In [0]:
control_table = f"{catalog_name}.{control_schema}.batch_control"

In [0]:
dbutils.widgets.text("batch_id","")
v_batch_id = dbutils.widgets.get("batch_id")

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import Row, functions as F

if v_batch_id:
    delta_table = DeltaTable.forName(spark, control_table)

    source_df = (
        spark.createDataFrame([Row(batch_id=v_batch_id, status="completed")])
        .withColumn("updated_at", F.current_timestamp())
    )

    (
        delta_table.alias("t")
        .merge(
            source_df.alias("s"),
            "t.batch_id = s.batch_id AND t.status ='in_progress'"
        )
        .whenMatchedUpdate(
            set={
                "status":"s.status",
                "updated_at":"s.updated_at"
            }
        )
        .execute()

    )
    print(f"Marked batch {v_batch_id} as completed")
else:
    raise Exception("batch_id is missing")